# 02 — Baseline Model (ResNet50)

Trains the paper baseline (ResNet50, 89.57% accuracy) on the ceilometer backscatter dataset,
with training tracked on **Weights & Biases**.

Assumes execution in Google Colab with the dataset under Google Drive
(`configs/config.yaml` → `dataset.path`), matching `notebooks/00-setup.ipynb`.

In [ ]:
import sys
sys.path.append("../src")

import yaml
import torch.nn as nn
import torch.optim as optim
import wandb

from dataset import get_dataloaders
from models import get_model, get_device, count_parameters
from evaluate import get_predictions, compute_metrics
from engine import train_one_epoch, evaluate_one_epoch, log_epoch_to_wandb

In [ ]:
wandb.login()

In [ ]:
with open("../configs/config.yaml") as f:
    cfg = yaml.safe_load(f)

config = {
    "model_name":    "resnet50",  # baseline — overrides config.yaml's model.name
    "dataset_path":  cfg["dataset"]["path"],
    "batch_size":    cfg["dataset"]["batch_size"],
    "image_size":    cfg["dataset"]["image_size"],
    "num_workers":   2,
    "num_classes":   cfg["model"]["num_classes"],
    "epochs":        cfg["training"]["epochs"],
    "optimizer":     cfg["training"]["optimizer"],
    "lr":            cfg["training"]["learning_rate"],
    "momentum":      cfg["training"]["momentum"],
    "weight_decay":  cfg["training"]["weight_decay"],
}

In [ ]:
run = wandb.init(
    project="cloud-detection",
    name="baseline-resnet50",
    job_type="train",
    config=config,
)
config = wandb.config

In [ ]:
device = get_device()

train_loader, val_loader, test_loader, class_names = get_dataloaders(
    dataset_path=config["dataset_path"],
    batch_size=config["batch_size"],
    image_size=config["image_size"],
    num_workers=config["num_workers"],
)

model = get_model(
    config["model_name"], num_classes=config["num_classes"], pretrained=True
).to(device)
count_parameters(model)

run.watch(model, log="all", log_freq=10)

In [ ]:
criterion = nn.CrossEntropyLoss()

if config["optimizer"] == "sgd":
    optimizer = optim.SGD(
        model.parameters(),
        lr=config["lr"],
        momentum=config["momentum"],
        weight_decay=config["weight_decay"],
    )
else:
    optimizer = optim.Adam(
        model.parameters(), lr=config["lr"], weight_decay=config["weight_decay"]
    )

## Training loop

`train_one_epoch` / `evaluate_one_epoch` / `log_epoch_to_wandb` live in `src/engine.py`
(shared with `03_new_model.ipynb`). Each epoch logs `train/loss`, `train/accuracy`,
`val/loss`, `val/accuracy`, and the validation confusion matrix to W&B.

In [ ]:
for epoch in range(1, config["epochs"] + 1):
    train_loss, train_acc, _, _ = train_one_epoch(
        model, train_loader, criterion, optimizer, device
    )
    val_loss, val_acc, val_preds, val_labels = evaluate_one_epoch(
        model, val_loader, criterion, device
    )

    log_epoch_to_wandb(
        epoch, train_loss, train_acc, val_loss, val_acc, val_preds, val_labels, class_names
    )

    print(
        f"Epoch {epoch:3d}/{config['epochs']}  "
        f"train_loss={train_loss:.4f} train_acc={train_acc:.4f}  "
        f"val_loss={val_loss:.4f} val_acc={val_acc:.4f}"
    )

## Final evaluation on the test set

In [ ]:
test_preds, test_labels = get_predictions(model, test_loader, device)
test_metrics = compute_metrics(test_preds, test_labels, model_name=config["model_name"])

wandb.log(
    {
        "test/accuracy": test_metrics["accuracy"],
        "test/f1": test_metrics["f1"],
        "test/precision": test_metrics["precision"],
        "test/recall": test_metrics["recall"],
        "test/confusion_matrix": wandb.plot.confusion_matrix(
            preds=test_preds,
            y_true=test_labels,
            class_names=class_names,
        ),
    }
)

run.finish()